# Étape 4 — DE CUDA v2 optimisé (issues #12 et #13)

**Prêt à exécuter : Exécution → Tout exécuter.** Runtime : **GPU T4**.

Le code est dans `src/cuda/de_cuda_v2.cu`.

## Ce qui change par rapport à la v1

| | v1 | v2 |
|---|---|---|
| Kernels par génération | 3 (mutation-croisement, évaluation, sélection) | **2** : kernel **P** (indices) + kernel fusionné **MCER** |
| Organisation | 1 thread par individu, boucle sur les D gènes | **1 bloc par individu**, 1 thread par gène |
| Accès mémoire | pas de D entre threads voisins (non coalescés) | gènes consécutifs entre threads voisins (**coalescés**) |
| Essai `trial` | tableau en mémoire globale | **mémoire partagée** du bloc |
| Évaluation f | 1 thread fait la boucle sur D | **réduction parallèle** dans le bloc (log₂ T étapes) |
| Remplacement | kernel séparé | dans MCER, écrit dans un **second tampon** (double tampon, puis échange des pointeurs) |
| Meilleur individu | N fitness copiées, min sur CPU | **argmin sur GPU**, 12 octets copiés |
| Threads par bloc | fixé (128) | **automatique** : puissance de 2 ≥ D, entre 32 et 256 |

Le double tampon est indispensable : dans MCER, un bloc lit x_r1, x_r2, x_r3 pendant que d'autres blocs écrivent leur gagnant. Écrire dans le tableau qu'on lit créerait une race condition.

## Critères
| Issue | Critère |
|---|---|
| #12 | même qualité que la v1 (Mann-Whitney p > 0,05) |
| #13 | tableau d'occupation + figure temps = f(N), v1 contre v2 |

In [ ]:
from google.colab import userdata
GH_USER = "amenacer"
REPO    = "DE_CUDA"
BRANCHE = "amina/4.1-4.5-optimiser-de-cuda-v2"
token   = userdata.get("GITHUB_TOKEN")
!rm -rf /content/{REPO}
!git clone -q https://{token}@github.com/{GH_USER}/{REPO}.git /content/{REPO}
%cd /content/DE_CUDA
!git config user.name  "Amina Menacer"
!git config user.email "aminamenacer9@gmail.com"
!git checkout -q {BRANCHE} && git branch --show-current
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import subprocess, re, statistics as st, os, json
ROOT = "/content/DE_CUDA"
RAPPORT = []                       # tout ce qui est affiche est aussi garde pour le rapport
def note(s=""):
    print(s); RAPPORT.append(str(s))

def run(args, cwd=ROOT, ok_codes=(0,)):
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if p.returncode not in ok_codes:
        print(p.stdout); print(p.stderr)
        raise RuntimeError(f"{args[0]} a renvoye {p.returncode}")
    return p

def compile_cuda(src, out):
    p = run(["nvcc", "-O3", "-arch=sm_75", "-Xcompiler", "-Wall", "-Xcompiler", "-Wextra",
             "-o", out, src], cwd=f"{ROOT}/src/cuda")
    warn = (p.stdout + p.stderr).strip()
    note(f"{src} : " + ("compilation SANS aucun warning" if not warn else "WARNINGS :\n" + warn))

KV = re.compile(r"(\w+)=(\S+)")
def lance(prog, f, D, N, seed, csv=None, extra=()):
    """Lance un run et renvoie le dictionnaire de sa derniere ligne (best_error, time_s, ...)."""
    args = [prog, f, str(D), str(N), str(seed)] + ([csv] if csv else []) + list(extra)
    out = run(args).stdout.strip().splitlines()[-1]
    d = dict(KV.findall(out))
    for k in ("best_error", "time_s"):
        d[k] = float(d[k])
    return d

def mann_whitney(a, b):
    """p-valeur du test de Mann-Whitney bilateral ; 1.0 si les deux series sont identiques et constantes."""
    from scipy.stats import mannwhitneyu
    if len(set(a) | set(b)) == 1:
        return 1.0
    return float(mannwhitneyu(a, b, alternative="two-sided").pvalue)

def tableau_md(lignes, entetes):
    s = "| " + " | ".join(entetes) + " |\n|" + "---|" * len(entetes) + "\n"
    for l in lignes:
        s += "| " + " | ".join(str(x) for x in l) + " |\n"
    return s
print("outils prets")

### 1 — Compiler les deux versions

In [ ]:
compile_cuda("de_cuda_v1.cu", "de_cuda_v1")
compile_cuda("de_cuda_v2.cu", "de_cuda_v2")
V1, V2 = f"{ROOT}/src/cuda/de_cuda_v1", f"{ROOT}/src/cuda/de_cuda_v2"

### 2 — Occupation théorique du kernel MCER selon la taille de bloc (#13)

`cudaOccupancyMaxActiveBlocksPerMultiprocessor` donne, pour chaque taille de bloc, combien de blocs tiennent en même temps sur un SM (limité par les registres et la mémoire partagée). « threads utiles » = part des threads qui ont un gène à traiter.

In [ ]:
import io, csv
rows = []
for D in (10, 50, 100):
    out = run([V2, "--occupation", str(D)]).stdout.strip().splitlines()
    note("\n".join(out[:2]))
    for r in csv.DictReader(io.StringIO("\n".join(out[2:]))):
        r = {"D": D, **r}; rows.append(r)
with open(f"{ROOT}/results/occupation_v2.csv", "w", newline="") as fp:
    w = csv.DictWriter(fp, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
note(tableau_md([[r["D"], r["threads/bloc"], r["shmem_octets"], r["blocs_par_SM"], r["occupation_%"], r["threads_utiles_%"]] for r in rows],
                ["D", "threads/bloc", "shmem (octets)", "blocs/SM", "occupation (%)", "threads utiles (%)"]))

### 3 — Contrôles de la v2 : 4 fonctions × D = 10 / 50 / 100

Chaque run vérifie lui-même que l'argmin GPU = min calculé sur CPU et que f(meilleur) par réduction parallèle = `evaluate()` du CPU. `sante=OK` sinon le run échoue. On vérifie aussi le déterminisme.

In [ ]:
lignes = []
for f in ["sphere", "rosenbrock", "griewank", "rastrigin"]:
    for D in (10, 50, 100):
        r = lance(V2, f, D, 100, 1)
        lignes.append([f, D, r["threads"], f"{r['best_error']:.3e}", r["FE"], f"{r['time_s']:.4f}", r["sante"]])
note(tableau_md(lignes, ["fonction", "D", "threads/bloc", "best_error", "FE", "temps (s)", "sante"]))
a = lance(V2, "rastrigin", 50, 100, 3); b = lance(V2, "rastrigin", 50, 100, 3)
note(f"Determinisme v2 (rastrigin D=50 N=100 seed 3) : {a['best_error']:.15e} / {b['best_error']:.15e} -> "
     + ("IDENTIQUES" if a["best_error"] == b["best_error"] else "DIFFERENTS"))

### 4 — Même qualité que la v1 ? (#12 : Mann-Whitney, 4 fonctions, D = 10, N = 50, 10 runs)

In [ ]:
CSV_Q = f"{ROOT}/results/qualite_v1_vs_v2.csv"
if os.path.exists(CSV_Q): os.remove(CSV_Q)
lignes = []
for f in ["sphere", "rosenbrock", "griewank", "rastrigin"]:
    e1 = [lance(V1, f, 10, 50, s, CSV_Q)["best_error"] for s in range(1, 11)]
    e2 = [lance(V2, f, 10, 50, s, CSV_Q)["best_error"] for s in range(1, 11)]
    p = mann_whitney(e1, e2)
    lignes.append([f, f"{st.mean(e1):.3e} ± {st.pstdev(e1):.2e}", f"{st.mean(e2):.3e} ± {st.pstdev(e2):.2e}",
                   f"{p:.3f}", "meme qualite" if p > 0.05 else "difference"])
note("Qualite v1 contre v2 (D = 10, N = 50, 10 runs, moyenne ± ecart-type de l'erreur)")
note(tableau_md(lignes, ["fonction", "v1", "v2", "p (Mann-Whitney)", "conclusion (p > 0,05)"]))

### 5 — Influence de la taille de bloc (#13) : Rastrigin, D = 100, N = 500, 3 runs

In [ ]:
CSV_T = f"{ROOT}/results/taille_bloc_v2.csv"
if os.path.exists(CSV_T): os.remove(CSV_T)
lignes = []
for T in (32, 64, 128, 256, 512):
    t = [lance(V2, "rastrigin", 100, 500, s, CSV_T, ["--threads", str(T)])["time_s"] for s in (1, 2, 3)]
    lignes.append([T, f"{st.mean(t):.4f} ± {st.pstdev(t):.4f}"])
note("Temps de la v2 selon le nombre de threads par bloc (choix automatique pour D = 100 : 128)")
note(tableau_md(lignes, ["threads/bloc", "temps (s)"]))

### 6 — Temps v1 contre v2 selon N et D (#13 : figure temps = f(N))

Rastrigin, N = 50 / 100 / 500, D = 10 / 50 / 100, 3 runs chacun.

In [ ]:
CSV_TV = f"{ROOT}/results/temps_v1_v2.csv"
if os.path.exists(CSV_TV): os.remove(CSV_TV)
TT = {}
lignes = []
for D in (10, 50, 100):
    for N in (50, 100, 500):
        t1 = [lance(V1, "rastrigin", D, N, s, CSV_TV)["time_s"] for s in (1, 2, 3)]
        t2 = [lance(V2, "rastrigin", D, N, s, CSV_TV)["time_s"] for s in (1, 2, 3)]
        TT[(D, N)] = (st.mean(t1), st.mean(t2))
        lignes.append([D, N, f"{st.mean(t1):.4f}", f"{st.mean(t2):.4f}", f"{st.mean(t1)/st.mean(t2):.2f}"])
note("Temps moyen (s), Rastrigin, 3 runs")
note(tableau_md(lignes, ["D", "N", "v1 (s)", "v2 (s)", "gain v1/v2"]))

In [ ]:
import matplotlib.pyplot as plt
os.makedirs(f"{ROOT}/figures", exist_ok=True)
Ns = [50, 100, 500]
fig, axs = plt.subplots(1, 3, figsize=(10, 3.3), sharey=True)
for ax, D in zip(axs, (10, 50, 100)):
    ax.plot(Ns, [TT[(D, N)][0] for N in Ns], "o-", label="v1 (3 kernels)")
    ax.plot(Ns, [TT[(D, N)][1] for N in Ns], "s-", label="v2 (P + MCER)")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xticks(Ns, [str(n) for n in Ns])
    ax.set_title(f"D = {D}"); ax.set_xlabel("N (population)")
axs[0].set_ylabel("temps (s), budget 10^4 x D"); axs[0].legend(frameon=False)
fig.suptitle("DE CUDA sur GPU T4, Rastrigin"); fig.tight_layout()
fig.savefig(f"{ROOT}/figures/temps_v1_v2_N.png", dpi=200); plt.show()

### 7 — Sauvegarder et pousser

In [ ]:
os.makedirs(f"{ROOT}/results", exist_ok=True)
with open(f"{ROOT}/results/etape4_v2_resultats.md", "w") as fp:
    fp.write("# Étape 4 — DE CUDA v2 : occupation, qualité, temps (issues #12 et #13)\n\n")
    fp.write("GPU : " + run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).stdout.strip() + "\n\n")
    fp.write("```\n" + "\n".join(RAPPORT) + "\n```\n")
%cd /content/DE_CUDA
!git add results/occupation_v2.csv results/qualite_v1_vs_v2.csv results/taille_bloc_v2.csv results/temps_v1_v2.csv figures/temps_v1_v2_N.png results/etape4_v2_resultats.md
!git status --short
!git commit -q -m "4.1-4.5 : resultats du DE CUDA v2 sur GPU T4 (refs #12 #13)" && git push -q origin {BRANCHE} && echo "=> pousse sur {BRANCHE}"